[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/07_metodos.ipynb)

# Ajustamento Básico - Modelos clássicos de ajustamento

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Apresentar e entender aplicações para o Modelo Condicionado

**Referência:** Gilbert Strang, Kai Borre (1997). *Linear Algebra, Geodesy, and GPS*. Wellesley-Cambridge Press. $\rightarrow$ **Cap 6, 7, 8 e 9**


## O Problema

Até aqui, o método dos mínimos quadrados (MMQ) foi sempre aplicado a um caso particular: as observações escritas explicitamente como função dos parâmetros desconhecidos, $F(X_a)=L_b$, o **Modelo Paramétrico**.

Esse, no entanto, é só um caso dentre três situações possíveis, que diferem em **o que aparece como incógnita** na equação de ajustamento:

| $\begin{matrix}\text{Situação}\end{matrix}$ | $\begin{matrix}\text{Equação}\end{matrix}$ | $\begin{matrix}\text{Modelo}\end{matrix}$ |
|---|---|---|
| As observações ajustadas podem ser escritas em função de parâmetros | $F(X_a) = L_b$ | **Paramétrico** |
| Não há parâmetros explícitos; as próprias observações ajustadas devem satisfazer uma condição geométrica/física | $F(L_a) = 0$ | **Condicionado** |
| Parâmetros e observações ajustadas aparecem juntos, sem que seja possível isolar um em função do outro | $F(X_a, L_a) = 0$ | **Combinado** |

## 2. Apresentar e entender aplicações para o Modelo Condicionado

No **Modelo Condicionado**, não se estimam parâmetros diretamente no ajustamento. O objetivo é corrigir as observações para que elas satisfaçam um conjunto de equações de condição independentes:

$$
F(L_a)=0.
$$

Como $L_a=L_b+V$, no caso linear escreve-se

$$
B V + W = 0,
$$

em que $B$ é a matriz das derivadas parciais das equações de condição em relação às observações e $W=F(L_b)$ é o vetor dos fechamentos, isto é, o quanto as observações brutas deixam de satisfazer as condições.

Aplicando o critério de mínimos quadrados com restrições, chega-se ao sistema em termos dos multiplicadores de Lagrange $K$:

$$
M K + W = 0,
$$

com

$$
M = B P^{-1} B^T.
$$

Logo,

$$
K=-M^{-1}W
$$

e os resíduos são obtidos por

$$
V=P^{-1}B^TK.
$$

Por fim,

$$
L_a=L_b+V.
$$

A redundância, ou número de graus de liberdade, é igual ao número de equações de condição independentes. A variância da unidade de peso a posteriori pode ser calculada por

$$
\hat\sigma_0^2=\frac{V^TPV}{m},
$$

onde $m$ é o número de equações de condição.

Para condições não lineares, o modelo é linearizado iterativamente. Na iteração $i$:

$$
B_iV_i + B_i(L_b-L_a^{i-1}) + F(L_a^{i-1})=0.
$$

Assim, o termo de fechamento usado na iteração é

$$
W_i=B_i(L_b-L_a^{i-1})+F(L_a^{i-1}).
$$

Calculam-se $V_i$ e $L_a^i=L_b+V_i$, repetindo o processo até que as correções se tornem desprezíveis. Uma vantagem desse modelo é que, em geral, não é necessário fornecer um chute inicial para parâmetros, pois o ajuste atua diretamente sobre as observações.

### Exemplo simples - Modelo Condicionado

Considere os três ângulos observados de um triângulo, com pesos iguais:

$$
L_b=\begin{bmatrix}60,1\\59,8\\60,4\end{bmatrix}^\circ.
$$

A condição geométrica é que a soma dos ângulos internos seja $180^\circ$:

$$
F(L_a)=L_{a1}+L_{a2}+L_{a3}-180=0.
$$

Como o modelo já é linear,

$$
B=\begin{bmatrix}1&1&1\end{bmatrix}, \qquad W=F(L_b)=60,1+59,8+60,4-180=0,3.
$$

Com $P=I$:

$$
M=BP^{-1}B^T=3.
$$

Logo,

$$
K=-M^{-1}W=-\frac{0,3}{3}=-0,1.
$$

Os resíduos são

$$
V=P^{-1}B^TK=\begin{bmatrix}-0,1\\-0,1\\-0,1\end{bmatrix}^\circ.
$$

Portanto,

$$
L_a=L_b+V=\begin{bmatrix}60,0\\59,7\\60,3\end{bmatrix}^\circ.
$$

Verificação:

$$
60,0+59,7+60,3=180,0^\circ.
$$

Como os pesos são iguais, o erro de fechamento $0,3^\circ$ foi distribuído igualmente entre as três observações.

### Exemplo — mesma rede de nivelamento, agora pelo Método Condicionado

Vamos refazer o exercício 8.9.1 do Gemael, mas agora **sem estimar parâmetros**: as próprias observações ajustadas devem satisfazer condições de fechamento de circuito.

**1º passo — quantas equações de condição?**

$$
\text{observações} - \text{incógnitas} = 9 - 5 = 4 \text{ equações de condição (graus de liberdade)}
$$

Um conjunto possível de condições (circuitos fechados da rede):

$$
F=\begin{cases}
\ell_{1a}+\ell_{2a}+\ell_{6a}-\ell_{8a}-(H_B-H_C)=0\\
-\ell_{9a}+\ell_{7a}-\ell_{8a}-(H_B-H_A)=0\\
\ell_{2a}+\ell_{3a}-\ell_{5a}=0\\
\ell_{3a}-\ell_{4a}+\ell_{7a}-\ell_{6a}=0
\end{cases}
$$

Note a **vantagem** do modelo condicionado: não é necessário um "chute" inicial $X_0$ para parâmetros — o ajuste atua diretamente sobre as observações.

In [ ]:
import numpy as np

# Dados do Exercício 8.9.1 (Gemael): Linha -> (desnível observado em m, distância em km)
dados = {
    1: (10.038, 1.14), 2: (8.297, 2.84), 3: (1.949, 3.21),
    4: (5.217, 6.03),  5: (10.244, 6.75), 6: (1.562, 0.84),
    7: (4.837, 2.94),  8: (3.370, 2.01), 9: (15.979, 5.28),
}
H_A, H_B, H_C = 33.831, 19.316, 2.791  # altitudes conhecidas

l = {i: dados[i][0] for i in dados}   # desníveis observados (Lb)
Lb_cond = np.array([l[i] for i in range(1,10)])
dist_cond = np.array([dados[i][1] for i in range(1,10)])
P_cond = np.diag(1.0/dist_cond)

# B = dF/dLa  (linhas = equações de condição; colunas = l1..l9)
#   F1: l1+l2+l6-l8-(HB-HC)=0
#   F2: -l9+l7-l8-(HB-HA)=0
#   F3: l2+l3-l5=0
#   F4: l3-l4+l7-l6=0
B = np.array([
    [1,1,0,0,0,1,0,-1,0],
    [0,0,0,0,0,0,1,-1,-1],
    [0,1,1,0,-1,0,0,0,0],
    [0,0,1,-1,0,-1,1,0,0],
])

W = np.array([
    l[1]+l[2]+l[6]-l[8]-(H_B-H_C),
    -l[9]+l[7]-l[8]-(H_B-H_A),
    l[2]+l[3]-l[5],
    l[3]-l[4]+l[7]-l[6],
])

M = B @ np.linalg.inv(P_cond) @ B.T
K = -np.linalg.solve(M, W)
V_cond = np.linalg.inv(P_cond) @ B.T @ K
La_cond = Lb_cond + V_cond

print("W (erros de fechamento, m):", np.round(W,4))
print("\nObservações ajustadas La (Método Condicionado):")
for i, val in enumerate(La_cond, start=1):
    print(f"  l{i}a = {val:.4f} m")

m_cond = B.shape[0]
sigma0_2_cond = (V_cond @ P_cond @ V_cond) / m_cond
print(f"\nsigma0^2 (condicionado) = {sigma0_2_cond:.8f}  (redundância m={m_cond})")

# Verificação: as observações ajustadas devem satisfazer EXATAMENTE as equações de condição
F_ajustado = B @ La_cond - np.array([H_B-H_C, H_B-H_A, 0, 0])
print(f"\nVerificação F(La) (deve ser ~0): {np.round(F_ajustado, 8)}")
print("Confere com o slide: l1a+l2a+l6a-l8a = HB-HC  ->",
      f"{La_cond[0]+La_cond[1]+La_cond[5]-La_cond[7]:.4f} = {H_B-H_C:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,4))
idx = np.arange(1,10)
ax.bar(idx, V_cond*1000, color='indianred')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(idx)
ax.set_xlabel('Linha (observação)')
ax.set_ylabel('Resíduo V (mm)')
ax.set_title('Resíduos por linha — Modelo Condicionado (Exercício 8.9.1)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Resíduos pequenos (sub-milimétricos a poucos milímetros) confirmam um bom ajuste,")
print("consistente com os erros de fechamento W originais (poucos milímetros).")

## Lista de exercícios complementares